In [25]:
# EDA e Visualização de Dados
import pandas as pd
import plotly.express as px

# ML 
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

# Otimização de Hiperparâmetros
import optuna


### Carregar Os Dados

In [19]:
# Carga de dados
df_clientes = pd.read_csv('../datasets/clients_datasets_pj.csv')


In [20]:
# Visualizar os dados
df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   atividade_economica     500 non-null    object 
 1   faturamento_mensal      500 non-null    float64
 2   numero_de_funcionarios  500 non-null    int64  
 3   localizacao             500 non-null    object 
 4   idade                   500 non-null    int64  
 5   inovacao                500 non-null    int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 23.6+ KB


In [ ]:
df_clientes.head(10)

,atividade_economica,faturamento_mensal,numero_de_funcionarios,localizacao,idade,inovacao
0,Comércio,713109.95,12,Rio de Janeiro,6,1
1,Comércio,790714.38,9,São Paulo,15,0
2,Comércio,1197239.33,17,São Paulo,4,9
3,Indústria,449185.78,15,São Paulo,6,0
4,Agronegócio,1006373.16,15,São Paulo,15,8
5,Serviços,1629562.41,16,Rio de Janeiro,11,4
6,Serviços,771179.95,13,Vitória,0,1
7,Serviços,707837.61,16,São Paulo,10,6
8,Comércio,888983.66,17,Belo Horizonte,10,1
9,Indústria,1098512.64,13,Rio de Janeiro,9,3


# EDA

In [30]:
# Distribuição da variavel  target
percetual_inovacao = df_clientes.value_counts('inovacao') / len(df_clientes) * 100
px.bar(percetual_inovacao, color=percetual_inovacao.index)

In [ ]:
# Teste ANOVA (Analysis of Variance)
# verifiar se ha varioacoes significativas ena media de faturamento mensal para diferentres niveis de inovacao
# Suposicao ou pressuposto:
## Observacoes indempendentes;
## variavel dependente 'e continua;
## segue uma distribuicao normal;
## homogeneidade de variancias.
## amostras sejam de tammanhos iguais ou semelhantes.


In [32]:
#vrificar se as variaveis (faturamento) entre os grupos  (inovacao) sao omogenias
## Aplicar testes de Bartlett;
## H0 - Variaveis sao iguais;
## H1 - Variaveis sao diferentes;

from scipy.stats import bartlett

#Separando as variaveis de faturamento em grupos pcomo base de coluna "inovacao"
dados_agrupados = [df_clientes['faturamento_mensal'][df_clientes['inovacao'] == grupo] for grupo in df_clientes['inovacao'].unique()]

#Excutando os resultados do teste de Bartlett
bartlett_test_statistic, bartlett_p_value = bartlett(*dados_agrupados)

# Exibindo os resultados
print(f"Estatística do Teste de Bartlett: {bartlett_test_statistic}")
print(f"Valor-p do Teste de Bartlett: {bartlett_p_value}")

Estatística do Teste de Bartlett: 10.901203117231173
Valor-p do Teste de Bartlett: 0.28254182954905804


In [33]:
# Executar o teste de Shapiro-Wilk para verificar a normalidade dos dados
# Verificar se os dados seguem uma distribuição normal
# H0 - Os dados seguem uma distribuicao normal;
# H1 - Os dados nao seguem uma distribuicao normal;

from scipy.stats import shapiro
 
# Executando o teste de Shapiro-Wilk para cada grupo
shapiro_test_statistics , shapiro_p_values = shapiro(df_clientes['faturamento_mensal'])

# Exibindo os resultados
print(f"Estatística do Teste de Shapiro-Wilk: {shapiro_test_statistics }")
print(f"Valor-p do Teste de Shapiro-Wilk: {shapiro_p_values }")

Estatística do Teste de Shapiro-Wilk: 0.9959857602472711
Valor-p do Teste de Shapiro-Wilk: 0.23513451034389005


In [36]:
# Aplicar a ANOVA de Welch, pos as amosreas sao de tamanhos diferentes;
## H0 nao ha diferencas significativas entre as medias dos grupos;
## H1 ha diferencas significativas entre as medias dos grupos;
from pingouin import welch_anova

analise_de_variancia = welch_anova(dv='faturamento_mensal', between='inovacao', data=df_clientes)

# Exibindo os resultados
print(f'Estatistica do teste de ANOVA Welch: {analise_de_variancia.loc[0, "F"]}')
print(f'Valor-p do teste de ANOVA Welch: {analise_de_variancia.loc[0, "p-unc"]}')


Estatistica do teste de ANOVA Welch: 1.1269836194061693
Valor-p do teste de ANOVA Welch: 0.34526211273911467


# Treinar o algoritimo K-Means

In [38]:
# Selecionar as colunas para clustering
x = df_clientes.copy()

# Separando variaveis numericas e categoricas e ordnais
numeric_features = ['faturamento_mensal', 'numero_de_funcionarios', 'idade']
categorical_features = ['localizacao', 'atividade_economica']
ordinal_features = ['inovacao']

# Aplicar transformacoes por tipo
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()
ordinal_transformer = OrdinalEncoder()

preprpcessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('ord', ordinal_transformer, ordinal_features)
    ]
)

# Transformar os dados
x_transformed = preprpcessor.fit_transform(x)

In [39]:
x_transformed

array([[-0.74634498, -0.54179191, -1.10058849, ...,  0.        ,
         0.        ,  1.        ],
       [-0.56165548, -1.5035527 ,  1.94344851, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.40582654,  1.06114274, -1.77704115, ...,  0.        ,
         0.        ,  9.        ],
       ...,
       [ 2.8196246 , -1.18296577,  0.25231684, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.03321411, -0.54179191, -1.43881482, ...,  0.        ,
         0.        ,  3.        ],
       [-2.03011486, -0.22120498, -1.77704115, ...,  1.        ,
         0.        ,  9.        ]], shape=(500, 12))

In [58]:
# Optuna para otimização de hiperparâmetros
def kmeans_objective(trial):
    
    # Ddefinir os hiperparâmetros a serem otimizados
    n_clusters = trial.suggest_int('n_clusters', 3, 10)
    distance_metric = trial.suggest_categorical('distance_metric', ['euclidean', 'minkowski'])

    # Criar o modelo
    modelo_kmeans = KMeans(n_clusters=n_clusters, random_state=51)

    # Treiar o Modelo
    modelo_kmeans.fit(x_transformed)

    #calcular o silhouette score
    distances = pairwise_distances(x_transformed, metric=distance_metric)
    silhouette_avg = silhouette_score(distances, modelo_kmeans.labels_)
    
    return silhouette_avg

In [59]:
# Criar o estudo do Op
seach_spacce = {'n_clusters': [3, 4, 5, 6, 7, 8, 9, 10], 'distance_metric': ['euclidean', 'minkowski']}
sampler = optuna.samplers.GridSampler(search_space=seach_spacce)
estudo_kmeans = optuna.create_study(direction='maximize', sampler=sampler)

# Rodar o estudo
estudo_kmeans.optimize(kmeans_objective, n_trials=100)

[I 2025-10-25 19:15:04,001] A new study created in memory with name: no-name-56faea98-afb1-4e47-8b0a-380969afd3f7
[I 2025-10-25 19:15:04,063] Trial 0 finished with value: 0.3847879496502923 and parameters: {'n_clusters': 4, 'distance_metric': 'euclidean'}. Best is trial 0 with value: 0.3847879496502923.
[I 2025-10-25 19:15:04,095] Trial 1 finished with value: 0.1123004838158908 and parameters: {'n_clusters': 9, 'distance_metric': 'euclidean'}. Best is trial 0 with value: 0.3847879496502923.
[I 2025-10-25 19:15:04,130] Trial 2 finished with value: 0.44454582909990875 and parameters: {'n_clusters': 3, 'distance_metric': 'minkowski'}. Best is trial 2 with value: 0.44454582909990875.
[I 2025-10-25 19:15:04,158] Trial 3 finished with value: 0.3847879496502923 and parameters: {'n_clusters': 4, 'distance_metric': 'minkowski'}. Best is trial 2 with value: 0.44454582909990875.
[I 2025-10-25 19:15:04,193] Trial 4 finished with value: 0.14731572416665714 and parameters: {'n_clusters': 8, 'distanc

In [60]:
# Melhor configuracao encontrada pelo Optuna
best_params = estudo_kmeans.best_params

# Intanciando o modelo K-Means com os melhores hiperparâmetros
best_kmeans = KMeans(n_clusters=best_params['n_clusters'], random_state=51)
best_kmeans.fit(x_transformed)

# Calculando o silhouette score 
distances = pairwise_distances(x_transformed, metric=best_params['distance_metric'])
best_silhouette_score = silhouette_score(distances, best_kmeans.labels_)

print(f'k (Numero de Clusters): {best_params["n_clusters"]}')
print(f'Metricas de Distancia Selecionada: {best_params["distance_metric"]}')
print(f'Silhouette Score: {best_silhouette_score}')

k (Numero de Clusters): 3
Metricas de Distancia Selecionada: euclidean
Silhouette Score: 0.4445458290999088


In [62]:
# Criar coluna com cluster escolhido
df_clientes['cluster'] = best_kmeans.labels_

In [63]:
# Visualizar os primeiros registros com os clusters
df_clientes.head(10)

,atividade_economica,faturamento_mensal,numero_de_funcionarios,localizacao,idade,inovacao,cluster
0,Comércio,713109.95,12,Rio de Janeiro,6,1,0
1,Comércio,790714.38,9,São Paulo,15,0,0
2,Comércio,1197239.33,17,São Paulo,4,9,1
3,Indústria,449185.78,15,São Paulo,6,0,0
4,Agronegócio,1006373.16,15,São Paulo,15,8,1
5,Serviços,1629562.41,16,Rio de Janeiro,11,4,2
6,Serviços,771179.95,13,Vitória,0,1,0
7,Serviços,707837.61,16,São Paulo,10,6,1
8,Comércio,888983.66,17,Belo Horizonte,10,1,0
9,Indústria,1098512.64,13,Rio de Janeiro,9,3,2


# Visualizar Resultados

In [64]:
# Cruzar idade e faturamento mensal por cluster
px.scatter(df_clientes, x='idade', y='faturamento_mensal', color='cluster' )

In [65]:
# Cruzar inovacao e faturamento mensal por cluster
px.scatter(df_clientes, x='inovacao', y='faturamento_mensal', color='cluster' )

In [69]:
# Cruzar numero de funcionarios e faturamento mensal por cluster
px.scatter(df_clientes, x='numero_de_funcionarios', y='faturamento_mensal', color='cluster' )

# Salvar o Modelo e o Pipeline de Transformacao

In [72]:
import joblib

# Salvar o Modelo de Transformacao
joblib.dump(best_kmeans, 'modelo_clusterizacao_clientes.pkl')

# Salvar o pipeline de transformacao
joblib.dump(preprpcessor, 'pipeline_clusterizacao_clientes.pkl')

['pipeline_clusterizacao_clientes.pkl']

# Aplicacao Batch no Gradio

In [77]:
import gradio as gr

modelo = joblib.load('./modelo_clusterizacao_clientes.pkl')
preprpcessor = joblib.load('./pipeline_clusterizacao_clientes.pkl')

def clustering(arquivo):
    # Carregar o arquivo CSV em um DataFrame
    df_empresas = pd.read_csv(arquivo.name)

    # Transformar os dados do DF para o formato que o KMeans precisa
    x_transformed = preprpcessor.transform(df_empresas)

    # Treinar modelo
    modelo.fit(x_transformed)

    # Criar coluna com cluster escolhido
    df_empresas['cluster'] = modelo.labels_
    df_empresas.to_csv('./clusters.csv', index=False)

    return './clusters.csv'

In [78]:
# Criar a interface do Gradio
app = gr.Interface(
    clustering,
    gr.File(file_types=[".csv"]),
    "file"
)

# Rodar a aplicacao
app.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "d:\projects\projecto_ia\kmeans_clustering\.venv\Lib\site-packages\uvicorn\protocols\http\h11_impl.py", line 403, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "d:\projects\projecto_ia\kmeans_clustering\.venv\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\projects\projecto_ia\kmeans_clustering\.venv\Lib\site-packages\fastapi\applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "d:\projects\projecto_ia\kmeans_clustering\.venv\Lib\site-packages\starlette\applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive,

Created dataset file at: .gradio\flagged\dataset1.csv
